In [1]:
import pandas as pd
from rouge_score import rouge_scorer
from sarathi.benchmark.request_generator.real_request_generator import RealRequestGenerator
from sarathi.benchmark.config import Config

/workspace/miniconda3/envs/vattn/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-05-19 03:38:29,135	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


unable to import module pod_attn


In [2]:
request_generator = RealRequestGenerator(config=Config({'num_requests': 20}))
cnn_prompts = request_generator.get_cnn_prompts()


In [3]:
scorer = rouge_scorer.RougeScorer(['rougeL', 'rouge1', 'rouge2'])

def get_rougeL_score(prompt, output):
    return scorer.score(prompt, output)['rougeL']

def get_avg_rougeL_score(prompts, outputs):
    precisions = []
    recalls = []
    fmeasures = []
    for prompt, output in zip(prompts, outputs):
        scores = get_rougeL_score(prompt, output)
        precisions.append(scores.precision)
        recalls.append(scores.recall)
        fmeasures.append(scores.fmeasure)
    return sum(precisions) / len(precisions), sum(recalls) / len(recalls), sum(fmeasures) / len(fmeasures)


In [15]:
policies = ["ee_nobatch", "off","eager", "lazy", "average", "rebatching"]
throughputs = []
precisions = []
recalls = []
fmeasures = []

ee_df = pd.read_csv(f"/workspace/xutingl/vattention-ee/outputs_70b/req_100_batch_4_csv/ee_batch1.csv")
avg_precision, avg_recall, avg_fmeasure = get_avg_rougeL_score(cnn_prompts, ee_df.output)
throughputs.append(ee_df.throughput.mean())
precisions.append(avg_precision)
recalls.append(avg_recall)
fmeasures.append(avg_fmeasure)

for policy in policies[1:]:
    df = pd.read_csv(f"/workspace/xutingl/vattention-ee/outputs_70b/req_100_batch_4_csv/{policy}.csv")
    avg_precision, avg_recall, avg_fmeasure = get_avg_rougeL_score(cnn_prompts, df.output)
    throughputs.append(df.throughput.mean())
    precisions.append(avg_precision)
    recalls.append(avg_recall)
    fmeasures.append(avg_fmeasure)
df = pd.DataFrame({
    "policy": policies,
    "throughput": throughputs,
    "precision": precisions,
    "recall": recalls,
    "fmeasure": fmeasures
})
df

,policy,throughput,precision,recall,fmeasure
0,ee_nobatch,26.282841,0.166531,0.143378,0.120703
1,off,71.546552,0.407214,0.241450,0.288385
2,eager,93.584302,0.025629,0.034102,0.028740
3,lazy,73.238791,0.382462,0.242704,0.279157
4,average,74.344407,0.359772,0.249517,0.279129
5,rebatching,80.815509,0.153599,0.112434,0.117615


In [9]:
policies = ["ee_nobatch", "off","eager", "lazy", "average", "rebatching"]
throughputs = []
rouge_scores = []

num_ee_tokens = []
num_no_ee_tokens = []
avg_conf_score = []
avg_conf_score_ee = []

ee_df = pd.read_csv(f"/workspace/xutingl/vattention-ee/outputs_70b/req_100_batch_4_csv/ee_batch1.csv")
avg_precision, avg_recall, avg_rouge_score = get_avg_rougeL_score(cnn_prompts, ee_df.output)
throughputs.append(ee_df.throughput.mean())
rouge_scores.append(avg_rouge_score)

num_ee_tokens.append(ee_df.num_ee_tokens.values[0])
num_no_ee_tokens.append(ee_df.num_no_ee_tokens.values[0])
avg_conf_score.append(ee_df.avg_conf_score.values[0])
avg_conf_score_ee.append(ee_df.avg_conf_score_ee.values[0])

for policy in policies[1:]:
    df = pd.read_csv(f"/workspace/xutingl/vattention-ee/outputs_70b/req_100_batch_4_csv/{policy}.csv")
    avg_precision, avg_recall, avg_rouge_score = get_avg_rougeL_score(cnn_prompts, df.output)
    throughputs.append(df.throughput.mean())
    rouge_scores.append(avg_rouge_score)

    num_ee_tokens.append(df.num_ee_tokens.values[0])
    num_no_ee_tokens.append(df.num_no_ee_tokens.values[0])
    avg_conf_score.append(df.avg_conf_score.values[0])
    avg_conf_score_ee.append(df.avg_conf_score_ee.values[0])
df = pd.DataFrame({
    "policy": policies,
    "throughput": throughputs,
    "ROUGE-L": rouge_scores,
    "num_ee_tokens": num_ee_tokens,
    "num_no_ee_tokens": num_no_ee_tokens,
    "avg_conf_score": avg_conf_score,
    "avg_conf_score_ee": avg_conf_score_ee
})
df

,policy,throughput,ROUGE-L,num_ee_tokens,num_no_ee_tokens,avg_conf_score,avg_conf_score_ee
0,ee_nobatch,26.552540,0.120703,32306,7970,0.908029,0.985430
1,off,72.612111,0.288385,0,1,0.757795,0.000000
2,eager,94.451514,0.028740,25432,169,0.919896,0.925298
3,lazy,73.143975,0.279157,444,16681,0.732171,0.952019
4,average,75.331483,0.279129,2344,16529,0.707134,0.903871
5,rebatching,81.450985,0.117615,14756,5145,0.792330,0.982013


In [6]:
policies = ["ee_nobatch", "off","eager", "lazy", "average", "rebatching"]
throughputs = []
rouge_scores = []
tpot = []

num_ee_tokens = []
num_no_ee_tokens = []
avg_conf_score = []
avg_conf_score_ee = []


ee_df = pd.read_csv(f"/workspace/xutingl/vattention-ee/outputs_70b/req_100_batch_4_csv/ee_batch1.csv")
avg_precision, avg_recall, avg_rouge_score = get_avg_rougeL_score(cnn_prompts, ee_df.output)
throughputs.append(ee_df.throughput.mean())
rouge_scores.append(avg_rouge_score)

tpot.append(ee_df.tpot.values[0])
num_ee_tokens.append(ee_df.num_ee_tokens.values[0])
num_no_ee_tokens.append(ee_df.num_no_ee_tokens.values[0])
avg_conf_score.append(ee_df.avg_conf_score.values[0])
avg_conf_score_ee.append(ee_df.avg_conf_score_ee.values[0])

for policy in policies[1:]:
    df = pd.read_csv(f"/workspace/xutingl/vattention-ee/outputs_70b/req_100_batch_4_csv/{policy}.csv")
    avg_precision, avg_recall, avg_rouge_score = get_avg_rougeL_score(cnn_prompts, df.output)
    throughputs.append(df.throughput.mean())
    rouge_scores.append(avg_rouge_score)

    tpot.append(df.tpot.values[0])
    num_ee_tokens.append(df.num_ee_tokens.values[0])
    num_no_ee_tokens.append(df.num_no_ee_tokens.values[0])
    avg_conf_score.append(df.avg_conf_score.values[0])
    avg_conf_score_ee.append(df.avg_conf_score_ee.values[0])
df = pd.DataFrame({
    "policy": policies,
    "throughput": throughputs,
    "TPOT": tpot,
    "ROUGE-L": rouge_scores,
    "num_ee_tokens": num_ee_tokens,
    "num_no_ee_tokens": num_no_ee_tokens,
    "avg_conf_score": avg_conf_score,
    "avg_conf_score_ee": avg_conf_score_ee
})
df

,policy,throughput,TPOT,ROUGE-L,num_ee_tokens,num_no_ee_tokens,avg_conf_score,avg_conf_score_ee
0,ee_nobatch,22.907277,0.042516,0.264900,8967,19312,0.793823,0.997913
1,off,69.853830,0.013548,0.270852,0,1,0.757795,0.000000
2,eager,86.979928,0.010960,0.074339,23904,865,0.908041,0.930441
3,lazy,70.640574,0.013431,0.270852,12,17041,0.764786,0.998544
4,average,69.879729,0.013535,0.293690,48,15765,0.752424,0.992550
5,rebatching,71.713645,0.013192,0.256510,4137,12119,0.667377,0.998350


In [ ]:
from matplotlib import pyplot as plt
import matplotlib.lines as mlines

# Define color and marker maps
policy_color = {
    'eager': '#bfa100',
    'average': 'green',
    'rebatching': 'blue',
}
threshold_marker = {
    '0.3': 'o',   # circle
    '0.5': '^',   # triangle
    '0.7': 's',   # square
}

plt.figure(figsize=(10, 6))

# Plot each point with the correct color and marker
for i, row in df.iterrows():
    policy_full = row['policy']
    # Parse policy and threshold
    if '_' in policy_full:
        policy, threshold = policy_full.split('_')
    else:
        policy, threshold = policy_full, ''
    color = policy_color.get(policy, 'gray')
    marker = threshold_marker.get(threshold, 'o')
    plt.scatter(row['ROUGE-L'], row['throughput'], color=color, marker=marker, s=100)
    plt.annotate(policy_full, (row['ROUGE-L'], row['throughput']),
                 textcoords="offset points", xytext=(5, 5), ha='left')

# Create custom legend
legend_elements = [
    mlines.Line2D([], [], color='#bfa100', marker='o', linestyle='None', markersize=10, label='eager'),
    mlines.Line2D([], [], color='green', marker='o', linestyle='None', markersize=10, label='average'),
    mlines.Line2D([], [], color='blue', marker='o', linestyle='None', markersize=10, label='rebatching'),
    mlines.Line2D([], [], color='black', marker='o', linestyle='None', markersize=10, label='thresh=0.3 (low)'),
    mlines.Line2D([], [], color='black', marker='^', linestyle='None', markersize=10, label='thresh=0.5 (medium)'),
    mlines.Line2D([], [], color='black', marker='s', linestyle='None', markersize=10, label='thresh=0.7 (high)'),
]
plt.legend(handles=legend_elements, loc='best', fontsize=10)

plt.xlabel("ROUGE-L")
plt.ylabel("Throughput")
plt.title("Throughput vs ROUGE-L for Different Policies")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [10]:
# No ee batch1

throughputs = []
rouge_scores = []

num_ee_tokens = []
num_no_ee_tokens = []
avg_conf_score = []
avg_conf_score_ee = []

ee_df = pd.read_csv(f"/workspace/xutingl/vattention-ee/outputs_70b/req_20_batch_4_csv/ee_batch1.csv")
avg_precision, avg_recall, avg_rouge_score = get_avg_rougeL_score(cnn_prompts, ee_df.output)
throughputs.append(ee_df.throughput.mean())
rouge_scores.append(avg_rouge_score)

num_ee_tokens.append(ee_df.num_ee_tokens.values[0])
num_no_ee_tokens.append(ee_df.num_no_ee_tokens.values[0])
avg_conf_score.append(ee_df.avg_conf_score.values[0])
avg_conf_score_ee.append(ee_df.avg_conf_score_ee.values[0])

print(throughputs, rouge_scores, num_ee_tokens, num_no_ee_tokens, avg_conf_score, avg_conf_score_ee)

[np.float64(21.612178278667827)] [0.29157343311008743] [np.int64(0)] [np.int64(1)] [np.float64(0.774136109742263)] [np.float64(0.0)]


In [7]:
# ee batch1 layer=60 conf=0.9

throughputs = []
rouge_scores = []

num_ee_tokens = []
num_no_ee_tokens = []
avg_conf_score = []
avg_conf_score_ee = []

ee_df = pd.read_csv(f"/workspace/xutingl/vattention-ee/outputs_70b/req_20_batch_4_csv/ee_batch1.csv")
avg_precision, avg_recall, avg_rouge_score = get_avg_rougeL_score(cnn_prompts, ee_df.output)
throughputs.append(ee_df.throughput.mean())
rouge_scores.append(avg_rouge_score)

num_ee_tokens.append(ee_df.num_ee_tokens.values[0])
num_no_ee_tokens.append(ee_df.num_no_ee_tokens.values[0])
avg_conf_score.append(ee_df.avg_conf_score.values[0])
avg_conf_score_ee.append(ee_df.avg_conf_score_ee.values[0])

print(throughputs, rouge_scores, num_ee_tokens, num_no_ee_tokens, avg_conf_score, avg_conf_score_ee)

[np.float64(25.914047819081084)] [0.16337160303943804] [np.int64(4763)] [np.int64(1512)] [np.float64(0.9143799224393138)] [np.float64(0.991260781249103)]


In [17]:
# ee batch1 layer=60 conf=0.99

throughputs = []
rouge_scores = []

num_ee_tokens = []
num_no_ee_tokens = []
avg_conf_score = []
avg_conf_score_ee = []

ee_df = pd.read_csv(f"/workspace/xutingl/vattention-ee/outputs_70b/req_20_batch_4_csv/ee_batch1.csv")
avg_precision, avg_recall, avg_rouge_score = get_avg_rougeL_score(cnn_prompts, ee_df.output)
throughputs.append(ee_df.throughput.mean())
rouge_scores.append(avg_rouge_score)

num_ee_tokens.append(ee_df.num_ee_tokens.values[0])
num_no_ee_tokens.append(ee_df.num_no_ee_tokens.values[0])
avg_conf_score.append(ee_df.avg_conf_score.values[0])
avg_conf_score_ee.append(ee_df.avg_conf_score_ee.values[0])

print(throughputs, rouge_scores, num_ee_tokens, num_no_ee_tokens, avg_conf_score, avg_conf_score_ee)

[np.float64(23.59758060594299)] [0.2648997506627636] [np.int64(2128)] [np.int64(3455)] [np.float64(0.832821910077708)] [np.float64(0.9987623090470644)]


In [ ]:
# ee batch1 layer=70 conf=0.9

throughputs = []
rouge_scores = []

num_ee_tokens = []
num_no_ee_tokens = []
avg_conf_score = []
avg_conf_score_ee = []

ee_df = pd.read_csv(f"/workspace/xutingl/vattention-ee/outputs_70b/req_20_batch_4_csv/ee_batch1.csv")
avg_precision, avg_recall, avg_rouge_score = get_avg_rougeL_score(cnn_prompts, ee_df.output)
throughputs.append(ee_df.throughput.mean())
rouge_scores.append(avg_rouge_score)

num_ee_tokens.append(ee_df.num_ee_tokens.values[0])
num_no_ee_tokens.append(ee_df.num_no_ee_tokens.values[0])
avg_conf_score.append(ee_df.avg_conf_score.values[0])
avg_conf_score_ee.append(ee_df.avg_conf_score_ee.values[0])

print(throughputs, rouge_scores, num_ee_tokens, num_no_ee_tokens, avg_conf_score, avg_conf_score_ee)

[np.float64(22.338862680347017)] [0.19982707037497188] [np.int64(1755)] [np.int64(3928)] [np.float64(0.7125145872798103)] [np.float64(0.9780423586524788)]


In [4]:
# ee batch1 layer=70 conf=0.99

throughputs = []
rouge_scores = []

num_ee_tokens = []
num_no_ee_tokens = []
avg_conf_score = []
avg_conf_score_ee = []

ee_df = pd.read_csv(f"/workspace/xutingl/vattention-ee/outputs_70b/req_20_batch_4_csv/ee_batch1.csv")
avg_precision, avg_recall, avg_rouge_score = get_avg_rougeL_score(cnn_prompts, ee_df.output)
throughputs.append(ee_df.throughput.mean())
rouge_scores.append(avg_rouge_score)

num_ee_tokens.append(ee_df.num_ee_tokens.values[0])
num_no_ee_tokens.append(ee_df.num_no_ee_tokens.values[0])
avg_conf_score.append(ee_df.avg_conf_score.values[0])
avg_conf_score_ee.append(ee_df.avg_conf_score_ee.values[0])

print(throughputs, rouge_scores, num_ee_tokens, num_no_ee_tokens, avg_conf_score, avg_conf_score_ee)

[np.float64(21.727027303448047)] [0.25193333578740595] [np.int64(638)] [np.int64(4064)] [np.float64(0.7133639960742507)] [np.float64(0.9974248556881488)]


In [12]:
# ee batch1 layer=75 conf=0.9

throughputs = []
rouge_scores = []

num_ee_tokens = []
num_no_ee_tokens = []
avg_conf_score = []
avg_conf_score_ee = []

ee_df = pd.read_csv(f"/workspace/xutingl/vattention-ee/outputs_70b/req_20_batch_4_csv/ee_batch1.csv")
avg_precision, avg_recall, avg_rouge_score = get_avg_rougeL_score(cnn_prompts, ee_df.output)
throughputs.append(ee_df.throughput.mean())
rouge_scores.append(avg_rouge_score)

num_ee_tokens.append(ee_df.num_ee_tokens.values[0])
num_no_ee_tokens.append(ee_df.num_no_ee_tokens.values[0])
avg_conf_score.append(ee_df.avg_conf_score.values[0])
avg_conf_score_ee.append(ee_df.avg_conf_score_ee.values[0])

print(throughputs, rouge_scores, num_ee_tokens, num_no_ee_tokens, avg_conf_score, avg_conf_score_ee)

[np.float64(22.01250905728256)] [0.25569117703886557] [np.int64(1818)] [np.int64(3210)] [np.float64(0.7180029574256853)] [np.float64(0.977161217050584)]


In [5]:
# ee batch1 layer=75 conf=0.9

throughputs = []
rouge_scores = []

num_ee_tokens = []
num_no_ee_tokens = []
avg_conf_score = []
avg_conf_score_ee = []

ee_df = pd.read_csv(f"/workspace/xutingl/vattention-ee/outputs_70b/req_20_batch_4_csv/ee_batch1.csv")
avg_precision, avg_recall, avg_rouge_score = get_avg_rougeL_score(cnn_prompts, ee_df.output)
throughputs.append(ee_df.throughput.mean())
rouge_scores.append(avg_rouge_score)

num_ee_tokens.append(ee_df.num_ee_tokens.values[0])
num_no_ee_tokens.append(ee_df.num_no_ee_tokens.values[0])
avg_conf_score.append(ee_df.avg_conf_score.values[0])
avg_conf_score_ee.append(ee_df.avg_conf_score_ee.values[0])

print(throughputs, rouge_scores, num_ee_tokens, num_no_ee_tokens, avg_conf_score, avg_conf_score_ee)

[np.float64(21.593651261684172)] [0.28654767752356214] [np.int64(924)] [np.int64(3591)] [np.float64(0.7362343212269185)] [np.float64(0.9973302993810538)]


In [16]:
# ee batch1 layer=76 conf=0.9

throughputs = []
rouge_scores = []

num_ee_tokens = []
num_no_ee_tokens = []
avg_conf_score = []
avg_conf_score_ee = []

ee_df = pd.read_csv(f"/workspace/xutingl/vattention-ee/outputs_70b/req_20_batch_4_csv/ee_batch1.csv")
avg_precision, avg_recall, avg_rouge_score = get_avg_rougeL_score(cnn_prompts, ee_df.output)
throughputs.append(ee_df.throughput.mean())
rouge_scores.append(avg_rouge_score)

num_ee_tokens.append(ee_df.num_ee_tokens.values[0])
num_no_ee_tokens.append(ee_df.num_no_ee_tokens.values[0])
avg_conf_score.append(ee_df.avg_conf_score.values[0])
avg_conf_score_ee.append(ee_df.avg_conf_score_ee.values[0])

print(throughputs, rouge_scores, num_ee_tokens, num_no_ee_tokens, avg_conf_score, avg_conf_score_ee)

[np.float64(22.011870237564956)] [0.2589751444458408] [np.int64(1968)] [np.int64(3017)] [np.float64(0.7251565490806882)] [np.float64(0.9778010903638068)]


In [14]:
# ee batch1 layer=77 conf=0.9

throughputs = []
rouge_scores = []

num_ee_tokens = []
num_no_ee_tokens = []
avg_conf_score = []
avg_conf_score_ee = []

ee_df = pd.read_csv(f"/workspace/xutingl/vattention-ee/outputs_70b/req_20_batch_4_csv/ee_batch1.csv")
avg_precision, avg_recall, avg_rouge_score = get_avg_rougeL_score(cnn_prompts, ee_df.output)
throughputs.append(ee_df.throughput.mean())
rouge_scores.append(avg_rouge_score)

num_ee_tokens.append(ee_df.num_ee_tokens.values[0])
num_no_ee_tokens.append(ee_df.num_no_ee_tokens.values[0])
avg_conf_score.append(ee_df.avg_conf_score.values[0])
avg_conf_score_ee.append(ee_df.avg_conf_score_ee.values[0])

print(throughputs, rouge_scores, num_ee_tokens, num_no_ee_tokens, avg_conf_score, avg_conf_score_ee)

[np.float64(21.801940985661144)] [0.3035184657043253] [np.int64(1453)] [np.int64(1323)] [np.float64(0.7825798724306504)] [np.float64(0.9816564859723684)]


In [13]:
# ee batch1 layer=78 conf=0.9

throughputs = []
rouge_scores = []

num_ee_tokens = []
num_no_ee_tokens = []
avg_conf_score = []
avg_conf_score_ee = []

ee_df = pd.read_csv(f"/workspace/xutingl/vattention-ee/outputs_70b/req_20_batch_4_csv/ee_batch1.csv")
avg_precision, avg_recall, avg_rouge_score = get_avg_rougeL_score(cnn_prompts, ee_df.output)
throughputs.append(ee_df.throughput.mean())
rouge_scores.append(avg_rouge_score)

num_ee_tokens.append(ee_df.num_ee_tokens.values[0])
num_no_ee_tokens.append(ee_df.num_no_ee_tokens.values[0])
avg_conf_score.append(ee_df.avg_conf_score.values[0])
avg_conf_score_ee.append(ee_df.avg_conf_score_ee.values[0])

print(throughputs, rouge_scores, num_ee_tokens, num_no_ee_tokens, avg_conf_score, avg_conf_score_ee)

[np.float64(21.872932199163763)] [0.2925876789363389] [np.int64(2371)] [np.int64(1569)] [np.float64(0.803737917919221)] [np.float64(0.9846866629084472)]
